In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

# Define the Autoregressive LSTM Model
class AutoregressiveLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=1):
        super(AutoregressiveLSTM, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # Initialize hidden state and cell state
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)

        # Forward pass through LSTM
        out, _ = self.lstm(x, (h0, c0))

        # Predict the next value in the sequence
        out = self.fc(out[:, -1, :])  # Use the last hidden state for prediction
        return out

# Hyperparameters
input_dim = 1  # Each input is a single value (e.g., a time step in a sequence)
hidden_dim = 64  # Hidden state dimension
output_dim = 1  # Predict a single value (next value in the sequence)
num_layers = 1  # Number of LSTM layers
sequence_length = 10  # Length of the input sequence
learning_rate = 0.001
num_epochs = 10
batch_size = 32

# Generate synthetic sequence data (e.g., a sine wave)
def generate_sine_wave(length, num_samples):
    x = np.linspace(0, 10, length * num_samples)
    y = np.sin(x)
    return y.reshape(num_samples, length, 1)  # Reshape to (num_samples, sequence_length, 1)

num_samples = 1000
data = generate_sine_wave(sequence_length + 1, num_samples)  # +1 for the target

# Prepare input and target sequences
inputs = data[:, :-1, :]  # All but the last time step
targets = data[:, -1, :]  # The last time step is the target

# Convert to PyTorch tensors
inputs = torch.tensor(inputs, dtype=torch.float32)
targets = torch.tensor(targets, dtype=torch.float32)

# Create DataLoader
dataset = TensorDataset(inputs, targets)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Initialize the model, loss function, and optimizer
model = AutoregressiveLSTM(input_dim, hidden_dim, output_dim, num_layers)
criterion = nn.MSELoss()  # Mean Squared Error Loss for regression
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Training loop
for epoch in range(num_epochs):
    for batch_inputs, batch_targets in dataloader:
        # Forward pass
        predictions = model(batch_inputs)
        loss = criterion(predictions, batch_targets)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

print("Training complete!")
